##### format_number()
- is used to **format numeric column** to a specified number of **decimal places**, returning the result as a **string**.

##### Syntax

     format_number(col, d)

|  Argument  | Description |
|------------|-------------|
|  **col**   | The column name **(string)** or Column object containing the **numeric values** to be formatted. |
|  **d**     | An **integer** representing the **number of decimal places** for **rounding**.                   |

In [0]:
from pyspark.sql.functions import col, format_number

##### 1) Basic Example: Format to 2 Decimal Places

- Automatically:
  - `Rounds values`
  - `Adds comma separators`
  - `Returns STRING` data type

In [0]:
data = [(1234.5678,), (9876.5,), (123456789.4567,)]

df = spark.createDataFrame(data, ["amount"])

df_frmt = df.select("amount",
         format_number("amount", 2).alias("formatted_amount"),
         format_number("amount", 0).alias("no_decimals"))
         
display(df_frmt)
df_frmt.printSchema()

amount,formatted_amount,no_decimals
1234.5678,"1,234.57","1,235"
9876.5,"9,876.50","9,876"
1.234567894567E8,"123,456,789.46","123,456,789"


root
 |-- amount: double (nullable = true)
 |-- formatted_amount: string (nullable = true)
 |-- no_decimals: string (nullable = true)



In [0]:
# Create a sample DataFrame
data = [(1, 123525.12345), (2, 13245123.6789), (3, None), (4, 5.0)]
columns = ["id", "num"]
df1 = spark.createDataFrame(data, columns)
display(df1)
df1.printSchema()

# Apply format_number to format the 'num' column
# Formats to 0 decimal places for large numbers and 4 for small numbers
df_formatted = df1 \
    .withColumn("formatted_num_0_decimals", format_number(col("num"), 0)) \
    .withColumn("formatted_num_4_decimals", format_number(col("num"), 4)) \
    .withColumn("formatted_num_7_decimals", format_number(col("num"), 7))

# Show the results
display(df_formatted)
df_formatted.printSchema()

id,num
1,123525.12345
2,1.32451236789E7
3,null
4,5.0


root
 |-- id: long (nullable = true)
 |-- num: double (nullable = true)



id,num,formatted_num_0_decimals,formatted_num_4_decimals,formatted_num_7_decimals
1,123525.12345,"123,525","123,525.1234","123,525.1234500"
2,1.32451236789E7,"13,245,124","13,245,123.6789","13,245,123.6789000"
3,null,null,null,null
4,5.0,5,5.0000,5.0000000


root
 |-- id: long (nullable = true)
 |-- num: double (nullable = true)
 |-- formatted_num_0_decimals: string (nullable = true)
 |-- formatted_num_4_decimals: string (nullable = true)
 |-- formatted_num_7_decimals: string (nullable = true)



##### 2) groupBy & aggregate

In [0]:
from pyspark.sql.functions import sum

sales_data = [
    ("A", 1000.456),
    ("A", 2000.789),
    ("B", 500.123),
    ("B", 3000.456),
    ("C", 1500.789),
    ("C", 2500.123),
    ("D", 3500.456)
]

df_agg = spark.createDataFrame(sales_data, ["category", "amount"])
display(df_agg)
df_agg.printSchema()

category,amount
A,1000.456
A,2000.789
B,500.123
B,3000.456
C,1500.789
C,2500.123
D,3500.456


root
 |-- category: string (nullable = true)
 |-- amount: double (nullable = true)



In [0]:
result = df_agg.groupBy("category").agg(sum("amount").alias("total"))
display(result)
result.printSchema()

category,total
A,3001.245
B,3500.579
C,4000.9120000000003
D,3500.456


root
 |-- category: string (nullable = true)
 |-- total: double (nullable = true)



In [0]:
result_final = result.select(
    "category", "total",
    format_number("total", 2).alias("formatted_total")
)

display(result_final)
result_final.printSchema()

category,total,formatted_total
A,3001.245,"3,001.24"
B,3500.579,"3,500.58"
C,4000.9120000000003,"4,000.91"
D,3500.456,"3,500.46"


root
 |-- category: string (nullable = true)
 |-- total: double (nullable = true)
 |-- formatted_total: string (nullable = true)



##### Common Mistake
- `After formatting, this will fail`.
- `Because it’s now a STRING`.
- If you need `numeric rounding (not string formatting)`.
      
      from pyspark.sql.functions import round

In [0]:
data = [(1234.5678,), (9876.5,), (123456789.4567,)]

df_str = spark.createDataFrame(data, ["amount"])
display(df_str)

amount
1234.5678
9876.5
1.234567894567E8


In [0]:
df_str.select("amount", format_number("amount", 2).alias("frmt_decimal")).display()

amount,frmt_decimal
1234.5678,"1,234.57"
9876.5,"9,876.50"
1.234567894567E8,"123,456,789.46"


In [0]:
# df_str.select("amount", format_number("amount", 2).alias("frmt_decimal")).display()
df_str.select("amount", format_number("amount", 2) + 100).alias("frmt_decimal").display()

---------------------------------------------------------------------------
NumberFormatException                     Traceback (most recent call last)
File <command-8376824141332046>, line 2
      1 # df_str.select("amount", format_number("amount", 2).alias("frmt_decimal")).display()
----> 2 df_str.select("amount", format_number("amount", 2) + 100).display()

File /databricks/python_shell/lib/dbruntime/monkey_patches.py:74, in apply_dataframe_display_patch.<locals>.df_display(df, *args, **kwargs)
     70 def df_display(df, *args, **kwargs):
     71     """
     72     df.display() is an alias for display(df). Run help(display) for more information.
     73     """
---> 74     display(df, *args, **kwargs)

File /databricks/python_shell/lib/dbruntime/display.py:133, in Display.display(self, input, *args, **kwargs)
    131     pass
    132 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 133     self.display_connect_table(input, **kwargs)
    134 elif isinsta

| Function          | Purpose           |
| ----------------- | ----------------- |
| `round()`         | Numeric rounding  |
| `bround()`        | Banker’s rounding |
| `format_number()` | String formatting |
| `format_string()` | Custom formatting |